In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
from scipy import sparse

# 1. Load the dataset
file_path = "perturb_processed.h5ad"
print(f"Loading {file_path}...")
adata = sc.read_h5ad(file_path)

# 2. Identify the missing perturbation names
# We look for conditions that are NOT 'ctrl' and NOT already in the gene list
conditions = adata.obs['condition'].unique()
existing_genes = set(adata.var['gene_name'])

missing_perts = []
for cond in conditions:
    if cond == 'ctrl':
        continue

    # Handle combinatorial cases (e.g., "DrugA+DrugB")
    parts = cond.split('+')
    for p in parts:
        if p not in existing_genes:
            missing_perts.append(p)

missing_perts = list(set(missing_perts)) # Unique list

if len(missing_perts) == 0:
    print("No missing perturbation genes found! The error might be elsewhere.")
else:
    print(f"Adding {len(missing_perts)} dummy genes: {missing_perts}")

    # 3. Create a matrix of Zeros for these new "genes"
    # Shape: (n_cells, n_missing_perts)
    n_cells = adata.shape[0]
    n_new = len(missing_perts)

    # Create sparse zero matrix
    new_X = sparse.csr_matrix((n_cells, n_new), dtype=np.float32)

    # 4. Create a temporary AnnData for the new genes
    obs_df = adata.obs.copy()
    var_df = pd.DataFrame(index=missing_perts)
    var_df['gene_name'] = missing_perts # Important: Fill the gene_name column

    # Fill other columns in var to match original (avoid concatenation errors)
    for col in adata.var.columns:
        if col not in var_df.columns:
            var_df[col] = "dummy" # or 0/False depending on type

    adata_dummy = sc.AnnData(X=new_X, obs=obs_df, var=var_df)

    # 5. Concatenate the original data with the dummy genes
    # axis=1 means we are adding columns (genes)
    adata_fixed = sc.concat([adata, adata_dummy], axis=1)

    # 6. Restore critical metadata that concat might drop
    # Ensure 'gene_name' is preserved and correct
    adata_fixed.var['gene_name'] = adata_fixed.var.index

    # Restore the uns dictionary (contains the rank_genes_groups stats we fixed earlier)
    adata_fixed.uns = adata.uns.copy()

    # 7. Final Check
    print(f"New shape: {adata_fixed.shape}")
    check_idx = np.where(adata_fixed.var['gene_name'] == missing_perts[0])[0]
    print(f"Test lookup for '{missing_perts[0]}': Found at index {check_idx}")

    # 8. Save
    print("Saving fixed file...")
    adata_fixed.write_h5ad(file_path)
    print("Success! You can now run the pert_data.load() command.")

In [ ]:
# 1. Load the dataset
file_path = "./data/project_dataset/perturb_processed.h5ad"
print(f"Loading {file_path}...")
adata = sc.read_h5ad(file_path)
adata